## 1. tokenizer, 构造输入
- tokenizer, model: 相匹配，tokenizer outputs => model input
- Auto*Tokenizer, AutoModel*：Generic type
- tokenizer：服务于 model input
   - len(input_ids) == len(attention_mask)
   - tokenizer(test_senteces[0], ): tokenizer.__call__：encode
   - tokenizer.encode == tokenizer.tokenize + tokenizer.convert_tokens_to_ids
   - tokenizer.decode
   - tokenizer 工作的原理其实就是 tokenizer.vocab：字典，存储了 token => id 的映射关系
       - tokenizer.special_tokens_map
   - attention mask 与 padding 相匹配；

In [1]:
test_senteces = ['today is not that bad', 'today is so bad', 'so good']
model_name = 'distilbert-base-uncased-finetuned-sst-2-english'

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [10]:
batch_input = tokenizer(test_senteces, truncation=True, padding=True, return_tensors="pt")
batch_input

{'input_ids': tensor([[ 101, 2651, 2003, 2025, 2008, 2919,  102],
        [ 101, 2651, 2003, 2061, 2919,  102,    0],
        [ 101, 2061, 2204,  102,    0,    0,    0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 0, 0, 0]])}

In [12]:
encode = tokenizer.encode(test_senteces[0])
encode

[101, 2651, 2003, 2025, 2008, 2919, 102]

In [13]:
tokenizer.decode(encode)

'[CLS] today is not that bad [SEP]'

In [15]:
tokenizer.vocab["[CLS]"]  # 字典

101

In [16]:
tokenizer.special_tokens_map.values()  # 特殊的标签

dict_values(['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]'])

In [17]:
tokenizer.convert_tokens_to_ids([special for special in tokenizer.special_tokens_map.values()])

[100, 102, 0, 101, 103]

## 2. model 模型调用

In [18]:
import torch
import torch.functional as F

In [19]:
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [20]:
model.config

DistilBertConfig {
  "_name_or_path": "distilbert-base-uncased-finetuned-sst-2-english",
  "activation": "gelu",
  "architectures": [
    "DistilBertForSequenceClassification"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "finetuning_task": "sst-2",
  "hidden_dim": 3072,
  "id2label": {
    "0": "NEGATIVE",
    "1": "POSITIVE"
  },
  "initializer_range": 0.02,
  "label2id": {
    "NEGATIVE": 0,
    "POSITIVE": 1
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "output_past": true,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "transformers_version": "4.27.1",
  "vocab_size": 30522
}

In [28]:
with torch.no_grad():
    output = model(**batch_input)
    print(output)
    labels = torch.argmax(output.logits,dim=1)
    print(labels)
    labels = [model.config.id2label[id] for id in labels.tolist()]
    print(labels)

SequenceClassifierOutput(loss=None, logits=tensor([[-3.4620,  3.6118],
        [ 4.7508, -3.7899],
        [-4.1938,  4.5566]]), hidden_states=None, attentions=None)
tensor([1, 0, 1])
['POSITIVE', 'NEGATIVE', 'POSITIVE']
